# Global Pollution Analysis and Energy Recovery
## Applied Machine Learning & Exploratory Data Analysis

### Project Overview
This project analyzes global environmental pollution (Air, Water, and Soil indices), industrial emissions, energy metrics, and macroeconomic indicators across 175 countries from 2000 to 2019. The dual modeling objectives are:
1. **Regression:** Predict waste-to-energy recovery in Gigawatt-hours (`Energy_Recovered (in GWh)`).
2. **Classification:** Predict ambient air pollution severity categorized into Low, Medium, and High classes without target leakage.

In this notebook, we follow strict data science discipline: ensuring data integrity, documenting anomalies, avoiding data leakage, and grounding all findings in empirical evidence.

---
### Notebook Outline
1. **Environment Setup & Configuration**
2. **Data Loading and Initial Inspection**
3. **Data Cleaning, Integrity & Anomaly Audit (Spain 2004 Investigation)**
4. **Descriptive Statistical Analysis**
5. **Exploratory Data Analysis (EDA) & Interpretations**
   - 5.1 Correlation Heatmap
   - 5.2 Pollution Distribution & Central Tendency (Boxplots)
   - 5.3 Pollution Trajectories Over Time (2000–2019)
   - 5.4 Cross-Country Pollution Comparisons
   - 5.5 Industrial & Plastic Waste Distributions
   - 5.6 Energy Recovery Target Distribution & Bivariate Relationships
6. **Summary of Phase 1 Findings & Methodology for Next Phases**

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Configure environment and styling
os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib'
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300

# Directory for persisted visual artifacts
VIZ_DIR = '../outputs/visualizations'
os.makedirs(VIZ_DIR, exist_ok=True)

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Visualizations directory: {os.path.abspath(VIZ_DIR)}")

Pandas version: 3.1.1
NumPy version: 2.5.4
Visualizations directory: /Users/paarthrana/ML-from-scratch/Global-Pollution-Analysis/outputs/visualizations


## 1. Data Loading and Initial Inspection
We load the raw dataset from `../data/raw/Global_Pollution_Analysis.csv`. We verify the dimensions, column types, and sample records directly from the file without assumptions.

In [2]:
DATA_PATH = '../data/raw/Global_Pollution_Analysis.csv'
df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()

print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns\n")
print("Column Information and Data Types:")
df.info()

Dataset Dimensions: 200 rows, 13 columns

Column Information and Data Types:
<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 13 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Country                                 200 non-null    str    
 1   Year                                    200 non-null    int64  
 2   Air_Pollution_Index                     200 non-null    float64
 3   Water_Pollution_Index                   200 non-null    float64
 4   Soil_Pollution_Index                    200 non-null    float64
 5   Industrial_Waste (in tons)              200 non-null    float64
 6   Energy_Recovered (in GWh)               200 non-null    float64
 7   CO2_Emissions (in MT)                   200 non-null    float64
 8   Renewable_Energy (%)                    200 non-null    float64
 9   Plastic_Waste_Produced (in tons)        200 non-null    float64
 

In [3]:
df.head(5)

,Country,Year,Air_Pollution_Index,Water_Pollution_Index,Soil_Pollution_Index,Industrial_Waste (in tons),Energy_Recovered (in GWh),CO2_Emissions (in MT),Renewable_Energy (%),Plastic_Waste_Produced (in tons),Energy_Consumption_Per_Capita (in MWh),Population (in millions),GDP_Per_Capita (in USD)
0,Hungary,2005,272.70,124.27,51.95,94802.83,158.14,5.30,41.11,37078.88,12.56,42.22,20972.96
1,Singapore,2001,86.72,60.34,117.22,56283.92,498.04,6.34,36.44,33128.20,5.23,137.25,34850.41
2,Romania,2016,91.59,83.36,121.72,56256.02,489.51,49.69,9.38,18803.46,13.15,124.47,57773.15
3,Cook Islands,2018,280.61,67.16,93.58,74864.73,145.18,8.91,18.97,9182.27,0.78,67.80,21837.51
4,Djibouti,2008,179.16,127.53,121.55,76862.06,40.38,14.93,34.00,39235.12,12.84,186.52,41379.37


## 2. Data Cleaning and Integrity Audit
Before computing summary statistics, we verify data integrity across three criteria:
1. **Missing / Null values:** Any unrecorded or corrupted entries.
2. **Full-row duplicate records:** Identical duplicate entries across all 13 attributes.
3. **Compound key uniqueness:** Checking whether `(Country, Year)` uniquely identifies each observation.

In [4]:
# 1. Missing Value Audit
missing_summary = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage (%)': (df.isnull().sum() / len(df)) * 100
})
print("=== Missing Value Audit ===")
print(missing_summary)

# 2. Exact Duplicate Rows
exact_dupes = df.duplicated().sum()
print(f"\nExact duplicate rows across all columns: {exact_dupes}")

# 3. Compound Key Duplicate Check (Country, Year)
cy_dupes = df.duplicated(subset=['Country', 'Year'], keep=False).sum()
print(f"Records sharing duplicate (Country, Year) keys: {cy_dupes}")

=== Missing Value Audit ===
                                        Missing_Count  Missing_Percentage (%)
Country                                             0                     0.0
Year                                                0                     0.0
Air_Pollution_Index                                 0                     0.0
Water_Pollution_Index                               0                     0.0
Soil_Pollution_Index                                0                     0.0
Industrial_Waste (in tons)                          0                     0.0
Energy_Recovered (in GWh)                           0                     0.0
CO2_Emissions (in MT)                               0                     0.0
Renewable_Energy (%)                                0                     0.0
Plastic_Waste_Produced (in tons)                    0                     0.0
Energy_Consumption_Per_Capita (in MWh)              0                     0.0
Population (in millions)            

### Spain 2004 Anomaly: Column-by-Column Investigation
While full-row duplicates are 0, exactly one compound key `(Spain, 2004)` appears twice in the dataset. Below, we conduct a column-by-column audit of both records.

In [5]:
spain_records = df[(df['Country'] == 'Spain') & (df['Year'] == 2004)]
audit_comparison = spain_records.T.rename(
    columns={spain_records.index[0]: 'Record A (Index 147)', spain_records.index[1]: 'Record B (Index 179)'}
)
audit_comparison['Difference / Note'] = audit_comparison.apply(
    lambda row: 'Identical' if row['Record A (Index 147)'] == row['Record B (Index 179)'] else 'Drastically Different', axis=1
)
audit_comparison

,Record A (Index 147),Record B (Index 179),Difference / Note
Country,Spain,Spain,Identical
Year,2004,2004,Identical
Air_Pollution_Index,217.42,134.92,Drastically Different
Water_Pollution_Index,97.5,157.31,Drastically Different
Soil_Pollution_Index,18.87,75.53,Drastically Different
Industrial_Waste (in tons),29480.82,88456.83,Drastically Different
Energy_Recovered (in GWh),54.68,307.91,Drastically Different
CO2_Emissions (in MT),43.72,2.42,Drastically Different
Renewable_Energy (%),31.11,18.68,Drastically Different
Plastic_Waste_Produced (in tons),49584.27,48265.22,Drastically Different


**Spain 2004 Investigation Findings & Decision:**
1. **Discrepancy across all numerical dimensions:** Every single numerical metric differs between Record A and Record B. For example:
   - Population differs by 40.23 million (105.76M vs. 145.99M).
   - GDP per capita differs by almost 2x ($49,171.17 vs. $26,353.46).
   - Industrial Waste differs by 3x (29,480.82 tons vs. 88,456.83 tons).
   - Energy Recovered differs by more than 5x (54.68 GWh vs. 307.91 GWh).
2. **Context & Demographic Plausibility:** Neither population figure corresponds to historical 2004 Spanish demographics (Spain's real population in 2004 was ~43.5 million; its nominal GDP per capita was ~$25,000 USD).
3. **Methodological Handling:** Without documented provenance indicating separate regional reporting or distinct survey methods, we **do not fabricate assumptions** or speculate. We do not automatically average or delete either observation. Both records are **retained initially** to preserve raw sample size ($N=200$). During predictive modeling, a sensitivity analysis will be noted to assess whether excluding either observation impacts model coefficients.

### Topology of the Dataset: Country Cardinality and Temporal Structure
We inspect the frequency of observations across countries to determine whether this dataset represents a longitudinal panel or pooled cross-sectional data.

In [6]:
n_unique_countries = df['Country'].nunique()
counts_per_country = df['Country'].value_counts()

print(f"Total Unique Countries: {n_unique_countries} out of {len(df)} total observations.")
print("\nBreakdown of observations per country:")
counts_per_country.value_counts().rename_axis("Observations per Country").reset_index(name="Number of Countries")

Total Unique Countries: 175 out of 200 total observations.

Breakdown of observations per country:


,Observations per Country,Number of Countries
0,1,152
1,2,21
2,3,2


**Methodological Decision on the `Country` Feature:**
- **152 out of 175 countries appear exactly once.** Only 21 countries appear twice, and 2 countries (Moldova and Latvia) appear three times.
- **High-Cardinality Limitation:** One-hot encoding `Country` would generate 174 sparse dummy columns for just 200 total samples (a feature-to-sample ratio exceeding $0.87$). This would cause catastrophic overfitting and multicollinearity.
- **Generalization:** A model relying on country dummies cannot generalize to unseen nations or out-of-sample data.
- **Decision:** Following senior data science discipline, `Country` is treated as an observational entity identifier and **excluded from the primary predictor set** in both regression and classification models.

## 3. Descriptive Statistical Analysis
We calculate the standard measures of central tendency, dispersion, quantiles, and skewness for all numerical features.

In [7]:
num_cols = df.select_dtypes(include=[np.number]).columns
desc_stats = df[num_cols].describe(percentiles=[0.25, 0.50, 0.75]).T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]
desc_stats['IQR'] = desc_stats['75%'] - desc_stats['25%']
desc_stats['Skewness'] = df[num_cols].skew()
desc_stats.round(2)

,count,mean,std,min,25%,50%,75%,max,IQR,Skewness
Year,200.0,2009.34,5.77,2000.00,2004.00,2010.00,2014.00,2019.00,10.00,-0.00
Air_Pollution_Index,200.0,180.63,67.07,50.30,134.97,183.38,237.42,297.95,102.45,-0.13
Water_Pollution_Index,200.0,115.07,47.58,31.13,74.55,112.30,157.48,199.32,82.93,0.05
Soil_Pollution_Index,200.0,76.49,39.69,11.15,40.89,78.60,109.21,149.23,68.32,0.03
Industrial_Waste (in tons),200.0,52891.68,27224.49,1019.37,31201.97,55299.15,74805.82,99739.36,43603.85,-0.09
Energy_Recovered (in GWh),200.0,260.45,147.14,11.73,118.35,273.14,384.96,499.98,266.60,-0.04
CO2_Emissions (in MT),200.0,24.88,14.47,1.92,11.22,25.36,38.55,49.69,27.33,0.03
Renewable_Energy (%),200.0,27.80,12.36,5.04,17.70,29.17,37.07,49.56,19.37,-0.09
Plastic_Waste_Produced (in tons),200.0,24492.89,14421.36,542.95,12843.88,24121.54,36516.23,49852.28,23672.35,0.03
Energy_Consumption_Per_Capita (in MWh),200.0,9.44,5.58,0.53,4.58,9.22,14.00,19.98,9.42,0.19


**Key Takeaways from Descriptive Statistics:**
- **Pollution Indices:** Air Pollution Index exhibits the widest absolute range (50.30 to 297.95, mean: 180.63), while Water Pollution Index averages 115.07 (31.13 to 199.32), and Soil Pollution Index averages 76.49 (11.15 to 149.23).
- **Target Variable (`Energy_Recovered (in GWh)`):** Spans evenly from 11.73 GWh to 499.98 GWh with a mean of 260.45 GWh and median of 273.56 GWh.
- **Skewness:** All numerical features exhibit skewness between $-0.20$ and $+0.25$, indicating symmetrical, well-bounded distributions without severe heavy tails that would necessitate log or power transformations.

## 4. Exploratory Data Analysis (EDA)
We construct publication-grade visualizations to uncover patterns, dependencies, temporal trajectories, and target relationships. All figures are generated with clear labels, legends, and unit notations, and persisted to `outputs/visualizations/`.

### 4.1 Correlation Analysis
We compute Pearson correlation coefficients across all numerical features to examine linear dependencies and check for multicollinearity.

In [8]:
plt.figure(figsize=(11, 8.5))
corr_matrix = df[num_cols].corr()

# Mask upper triangle for visual clarity
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

clean_labels = [c.replace(' (in tons)', '').replace(' (in GWh)', '').replace(' (in MT)', '')
                 .replace(' (in MWh)', '').replace(' (in millions)', '').replace(' (in USD)', '') 
                for c in num_cols]

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-0.3, vmax=0.3, center=0, linewidths=0.5,
            xticklabels=clean_labels, yticklabels=clean_labels,
            cbar_kws={'label': 'Pearson Correlation Coefficient (r)'})

plt.title('Correlation Heatmap of Numerical Features', fontsize=14, weight='bold', pad=15)
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/correlation_heatmap.png', dpi=300)
plt.show()

/var/folders/rq/9dtg1ypn7gnb8c1mm2f8xq680000gn/T/ipykernel_55573/3824732429.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation of Correlation Heatmap:**
1. **Target Correlations:** `Energy_Recovered (in GWh)` exhibits very low linear correlation with all candidate predictors:
   - Industrial Waste: $r = -0.16$ (weak negative correlation)
   - Year: $r = +0.09$
   - Soil Pollution Index: $r = +0.06$
   - Renewable Energy (%): $r = +0.04$
   - CO2 Emissions: $r = +0.02$
   - Air Pollution Index: $r = +0.003$
   - Energy Consumption Per Capita: $r = -0.03$
2. **Absence of Multicollinearity:** No pair of numerical predictors has a correlation coefficient $|r| > 0.30$. The highest observed correlations are $r = +0.24$ (Water Pollution Index and Population) and $r = -0.22$ (Renewable Energy and GDP Per Capita).
3. **Data Science Reality:** In an honest analysis, weak linear correlations indicate that Linear Regression will have modest explanatory power ($R^2$ close to 0). It is essential to benchmark the model against a dummy mean baseline rather than over-interpreting noise.

### 4.2 Pollution Distribution & Central Tendency
We compare the distributions and spreads of `Air_Pollution_Index`, `Water_Pollution_Index`, and `Soil_Pollution_Index` using box plots.

In [9]:
plt.figure(figsize=(9, 5.5))
palette = {'Air_Pollution_Index': '#e74c3c', 'Water_Pollution_Index': '#3498db', 'Soil_Pollution_Index': '#2ecc71'}
sns.boxplot(data=df[['Air_Pollution_Index', 'Water_Pollution_Index', 'Soil_Pollution_Index']], 
            palette=palette, width=0.45, flierprops={'marker': 'o', 'markersize': 5, 'markerfacecolor': 'grey'})

plt.title('Distribution Comparison of Pollution Indices', fontsize=14, weight='bold', pad=15)
plt.ylabel('Index Score', fontsize=12)
plt.xticks([0, 1, 2], ['Air Pollution Index', 'Water Pollution Index', 'Soil Pollution Index'], fontsize=11)
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/pollution_boxplots.png', dpi=300)
plt.show()

/var/folders/rq/9dtg1ypn7gnb8c1mm2f8xq680000gn/T/ipykernel_55573/1977928728.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation of Pollution Box Plots:**
- `Air_Pollution_Index` has the highest central tendency (median $\approx 184$, IQR $\approx 103$), spanning from 50 to nearly 300.
- `Water_Pollution_Index` is situated in the intermediate range (median $\approx 113$, IQR $\approx 83$), spanning roughly 30 to 200.
- `Soil_Pollution_Index` is on the lowest scale (median $\approx 79$, IQR $\approx 69$), bounded between 11 and 149.
- No points fall outside $1.5 \times \text{IQR}$, indicating zero statistical outliers in any of the three pollution indices.

### 4.3 Pollution Trends Over Time (2000–2019)
We evaluate the annual trajectories of the three pollution indices across the 20-year observation window.

In [10]:
plt.figure(figsize=(11, 6))
yearly = df.groupby('Year')[['Air_Pollution_Index', 'Water_Pollution_Index', 'Soil_Pollution_Index']].mean().reset_index()

plt.plot(yearly['Year'], yearly['Air_Pollution_Index'], marker='o', linewidth=2.5, label='Air Pollution Index', color='#e74c3c')
plt.plot(yearly['Year'], yearly['Water_Pollution_Index'], marker='s', linewidth=2.5, label='Water Pollution Index', color='#3498db')
plt.plot(yearly['Year'], yearly['Soil_Pollution_Index'], marker='^', linewidth=2.5, label='Soil Pollution Index', color='#2ecc71')

plt.title('Annual Mean Pollution Indices (2000–2019)', fontsize=14, weight='bold', pad=15)
plt.xlabel('Year', fontsize=12, labelpad=10)
plt.ylabel('Mean Pollution Index Score', fontsize=12, labelpad=10)
plt.xticks(yearly['Year'], rotation=45)
plt.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/pollution_trends_over_time.png', dpi=300)
plt.show()

/var/folders/rq/9dtg1ypn7gnb8c1mm2f8xq680000gn/T/ipykernel_55573/3208729199.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation of Pollution Trends Over Time:**
- Annual means oscillate within stable boundaries across all two decades (Air: 155–205; Water: 95–135; Soil: 65–90).
- There is no discernible secular global trend (neither persistent improvement nor degradation).
- The annual fluctuations reflect the shifting sample of countries represented in each year (~5 to 15 different nations per year), rather than a synchronized macro-environmental cycle.

### 4.4 Country Pollution Comparison
We examine the 10 lowest and 10 highest countries ranked by their recorded mean Air Pollution Index.

In [11]:
country_air = df.groupby('Country')['Air_Pollution_Index'].mean().sort_values()
top_bottom_countries = pd.concat([country_air.head(10), country_air.tail(10)])

plt.figure(figsize=(12, 7))
colors = ['#2ecc71'] * 10 + ['#e74c3c'] * 10
plt.barh(top_bottom_countries.index, top_bottom_countries.values, color=colors, edgecolor='black', alpha=0.85)

global_mean = df['Air_Pollution_Index'].mean()
plt.axvline(global_mean, color='black', linestyle='--', linewidth=1.5, label=f'Global Mean ({global_mean:.1f})')

plt.title('Country Comparison: Lowest 10 vs. Highest 10 Mean Air Pollution Index', fontsize=14, weight='bold', pad=15)
plt.xlabel('Mean Air Pollution Index', fontsize=12, labelpad=10)
plt.ylabel('Country', fontsize=12)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/country_pollution_comparison.png', dpi=300)
plt.show()

/var/folders/rq/9dtg1ypn7gnb8c1mm2f8xq680000gn/T/ipykernel_55573/3943100148.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation of Country Comparisons:**
- The lowest recorded air pollution scores (e.g., bottom 10) range between 50 and 80.
- The highest recorded air pollution scores (e.g., top 10) exceed 270.
- **Statistical Caution:** Because 152 of the 175 countries appear only once in the dataset, these country comparisons reflect individual historical sample records rather than long-term longitudinal country baselines. Rankings should not be interpreted as permanent national performance indices.

### 4.5 Industrial and Plastic Waste Distributions
We inspect the distribution, variance, and density of industrial and plastic waste.

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['Industrial_Waste (in tons)'], kde=True, ax=axes[0], color='#8e44ad', bins=15)
axes[0].set_title('Industrial Waste Distribution', fontsize=13, weight='bold')
axes[0].set_xlabel('Industrial Waste (tons)', fontsize=11)
axes[0].set_ylabel('Observation Count', fontsize=11)

sns.histplot(df['Plastic_Waste_Produced (in tons)'], kde=True, ax=axes[1], color='#16a085', bins=15)
axes[1].set_title('Plastic Waste Produced Distribution', fontsize=13, weight='bold')
axes[1].set_xlabel('Plastic Waste (tons)', fontsize=11)
axes[1].set_ylabel('Observation Count', fontsize=11)

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/industrial_waste_distribution.png', dpi=300)
plt.show()

/var/folders/rq/9dtg1ypn7gnb8c1mm2f8xq680000gn/T/ipykernel_55573/1747168774.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation of Waste Distributions:**
- Industrial waste (1,000 to 100,000 tons) and plastic waste (500 to 50,000 tons) exhibit uniform-like dispersion across their operational ranges.
- There are no severe zero-inflation artifacts, clipping boundaries, or extreme long tails.

### 4.6 Energy Recovery Target Distribution & Bivariate Relationships
We examine the distribution of `Energy_Recovered (in GWh)` and its relationships with key industrial and pollution covariates.

In [13]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(df['Energy_Recovered (in GWh)'], kde=True, ax=axes[0], color='#27ae60', bins=15)
axes[0].set_title('Energy Recovered (GWh) - Histogram & KDE', fontsize=13, weight='bold')
axes[0].set_xlabel('Energy Recovered (GWh)', fontsize=11)
axes[0].set_ylabel('Observation Count', fontsize=11)

sns.boxplot(x=df['Energy_Recovered (in GWh)'], ax=axes[1], color='#a9dfbf', width=0.3)
axes[1].set_title('Energy Recovered (GWh) - Boxplot', fontsize=13, weight='bold')
axes[1].set_xlabel('Energy Recovered (GWh)', fontsize=11)

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/energy_recovery_distribution.png', dpi=300)
plt.show()

/var/folders/rq/9dtg1ypn7gnb8c1mm2f8xq680000gn/T/ipykernel_55573/1876254973.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

bivariate_features = [
    ('Industrial_Waste (in tons)', 'Industrial Waste (tons)', '#8e44ad'),
    ('CO2_Emissions (in MT)', 'CO2 Emissions (MT)', '#d35400'),
    ('Air_Pollution_Index', 'Air Pollution Index', '#c0392b'),
    ('Energy_Consumption_Per_Capita (in MWh)', 'Energy Consumption Per Capita (MWh)', '#2980b9')
]

for ax, (col, label, clr) in zip(axes.flatten(), bivariate_features):
    sns.regplot(data=df, x=col, y='Energy_Recovered (in GWh)', ax=ax,
                color=clr, scatter_kws={'alpha': 0.6, 'edgecolor': 'k', 's': 45},
                line_kws={'color': 'black', 'linewidth': 1.5, 'linestyle': '--'})
    r_val = df[col].corr(df['Energy_Recovered (in GWh)'])
    ax.set_title(f'Energy Recovered vs. {label}\n(Pearson r = {r_val:.3f})', fontsize=12, weight='bold')
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel('Energy Recovered (GWh)', fontsize=11)

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/energy_recovery_relationships.png', dpi=300)
plt.show()

/var/folders/rq/9dtg1ypn7gnb8c1mm2f8xq680000gn/T/ipykernel_55573/502355710.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation of Energy Recovery Relationships:**
- `Energy_Recovered (in GWh)` is uniformly distributed between 11.7 GWh and 500 GWh (Mean: 260.5 GWh, Median: 273.6 GWh).
- Fitted OLS regression lines against Industrial Waste ($r = -0.161$), CO2 Emissions ($r = 0.025$), Air Pollution Index ($r = 0.003$), and Per Capita Energy Consumption ($r = -0.030$) demonstrate virtually horizontal slopes.
- This empirical evidence confirms that energy recovery is not linearly determined by individual pollution or waste features alone, setting realistic performance expectations for linear modeling.

## 5. Phase 1 Summary and Transition to Phase 2

### Summary of Completed Work in Phase 1:
1. **Data Integrity:** The raw dataset of 200 rows and 13 columns has zero missing values.
2. **Anomaly Documented:** The Spain 2004 duplicate key was investigated column-by-column. Both entries have been retained without speculative regional assumptions, preserving the raw sample size.
3. **Cardinality Handling:** We established that `Country` exhibits 175 categories across 200 records (pooled cross-sectional data). To avoid high-cardinality overfitting and poor out-of-sample generalization, `Country` is treated as an observational identifier rather than a primary regressor.
4. **Descriptive Statistics & Visualizations:** Completed 7 comprehensive visual analyses (Correlation Heatmap, Pollution Boxplots, Annual Trajectories, Country Comparisons, Waste Distributions, and Target Bivariate Plots), with all artifacts saved to `outputs/visualizations/`.
5. **Linearity Realities:** We confirmed that linear correlations with `Energy_Recovered (in GWh)` are weak ($|r| < 0.2$), highlighting the necessity of benchmarking against a dummy mean baseline.

### Plan for Phase 2 (Feature Engineering) & Beyond:
- **Total Energy Consumption Calculation:** Apply verified physical formula:
  $$\text{Total Energy Consumption (GWh)} = \text{Energy\_Consumption\_Per\_Capita (MWh/person)} \times \text{Population (millions)} \times 1000$$
  and compare performance with and without this feature.
- **Severity Classification Target Definition:** Compute tertile thresholds strictly on the training partition ($Q_{33\%}, Q_{66\%}$) to assign Low, Medium, and High classes.
- **Strict Leakage Prevention:** Remove `Air_Pollution_Index` completely from classification feature matrix $X$.
- **Dual Splitting Protocol:** Prepare both 80/20 Random and 2000–2014 vs. 2015–2019 Chronological splits.